# 03 - Gobernanza con Unity Catalog

Este notebook documenta y demuestra la gobernanza del proyecto mediante Unity Catalog.

Objetivos:

- verificar el catálogo propio del proyecto y sus schemas Medallion;
- identificar el principal que ejecuta el notebook sin hardcodear usuarios;
- documentar un esquema de permisos de solo lectura sobre Silver;
- aplicar privilegios `USE CATALOG`, `USE SCHEMA` y `SELECT`;
- verificar los permisos mediante `SHOW GRANTS`;
- dejar evidencia reproducible de gobernanza para la rúbrica del proyecto.


## 1. Configuración

Se reutiliza el catálogo creado para el proyecto y los tres schemas de la arquitectura Medallion.

In [ ]:
dbutils.widgets.text("catalog_name", "accidentalidad_databricks", "Catalog del proyecto")

PROY_CATALOG = dbutils.widgets.get("catalog_name")
PROY_SCHEMA_BRONZE = "bronze"
PROY_SCHEMA_SILVER = "silver"
PROY_SCHEMA_GOLD = "gold"

print(f"Catalog: {PROY_CATALOG}")
print(f"Schemas: {PROY_SCHEMA_BRONZE}, {PROY_SCHEMA_SILVER}, {PROY_SCHEMA_GOLD}")


## 2. Verificación de la estructura gobernada

El proyecto utiliza un catálogo propio y separa los datos por nivel de refinamiento. Esta separación facilita aplicar permisos distintos por capa.

In [ ]:
print("Schemas disponibles en el catálogo:")
display(spark.sql(f"SHOW SCHEMAS IN {PROY_CATALOG}"))


In [ ]:
for schema in [PROY_SCHEMA_BRONZE, PROY_SCHEMA_SILVER, PROY_SCHEMA_GOLD]:
    print(f"\nTablas en {PROY_CATALOG}.{schema}:")
    display(spark.sql(f"SHOW TABLES IN {PROY_CATALOG}.{schema}"))


## 3. Identificación dinámica del principal

No se hardcodea ningún correo, usuario o grupo. `current_user()` devuelve el principal que está ejecutando el notebook.

En un entorno empresarial real, los privilegios normalmente se asignarían a grupos administrados por identidad, por ejemplo un grupo de analistas o ingenieros. Para este proyecto individual se utiliza el principal actual únicamente para demostrar el mecanismo de Unity Catalog.

In [ ]:
CURRENT_PRINCIPAL = spark.sql("SELECT current_user() AS principal").first()["principal"]

print(f"Principal actual: {CURRENT_PRINCIPAL}")


## 4. Política de acceso demostrada

Se aplica un patrón de **solo lectura sobre Silver**:

1. `USE CATALOG` permite resolver objetos dentro del catálogo.
2. `USE SCHEMA` permite acceder al namespace `silver`.
3. `SELECT` permite consultar las tablas del schema.

Esto ilustra el principio de mínimo privilegio: para consumir datos Silver no es necesario otorgar permisos de escritura ni administración.

In [ ]:
# Escapar backticks por seguridad antes de interpolar el principal en SQL.
principal_sql = CURRENT_PRINCIPAL.replace('`', '``')

sentencias_grant = [
    f"GRANT USE CATALOG ON CATALOG {PROY_CATALOG} TO `{principal_sql}`",
    f"GRANT USE SCHEMA ON SCHEMA {PROY_CATALOG}.{PROY_SCHEMA_SILVER} TO `{principal_sql}`",
    f"GRANT SELECT ON SCHEMA {PROY_CATALOG}.{PROY_SCHEMA_SILVER} TO `{principal_sql}`"
]

for sentencia in sentencias_grant:
    print(sentencia)
    spark.sql(sentencia)

print("[OK] Privilegios aplicados.")


## 5. Evidencia de permisos

Se consultan los grants del catálogo y del schema Silver para dejar evidencia del control de acceso.

In [ ]:
print(f"Grants sobre catálogo {PROY_CATALOG}:")
display(spark.sql(f"SHOW GRANTS ON CATALOG {PROY_CATALOG}"))


In [ ]:
print(f"Grants sobre schema {PROY_CATALOG}.{PROY_SCHEMA_SILVER}:")
display(spark.sql(
    f"SHOW GRANTS ON SCHEMA {PROY_CATALOG}.{PROY_SCHEMA_SILVER}"
))


## 6. Validación de acceso de lectura

Se ejecuta una consulta sencilla sobre una tabla Silver para comprobar que el principal puede consumir la capa gobernada.

In [ ]:
resultado = spark.sql(f"""
    SELECT
        COUNT(*) AS total_accidentes,
        COUNT(DISTINCT croquis) AS croquis_distintos,
        MIN(fecha_accidente) AS fecha_minima,
        MAX(fecha_accidente) AS fecha_maxima
    FROM {PROY_CATALOG}.{PROY_SCHEMA_SILVER}.accidentes
""")

display(resultado)


## 7. Resumen de gobernanza

La gobernanza del proyecto queda demostrada mediante:

- un catálogo dedicado: `accidentalidad_databricks`;
- schemas separados para Bronze, Silver y Gold;
- identificación dinámica del principal;
- permisos explícitos de uso y lectura sobre Silver;
- verificación con `SHOW GRANTS`;
- validación de acceso mediante una consulta real.

En producción, estos privilegios deberían asignarse preferiblemente a grupos y no directamente a usuarios individuales.